In [2]:
!pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl (20 kB)


You should consider upgrading via the 'C:\Users\User\Downloads\nusrat\medrag\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
import pandas as pd
import json
import asyncio
import aiohttp
import re
import os
from dotenv import load_dotenv
from pathlib import Path
load_dotenv()

True

In [4]:
# Test sample
test_sample = {
    "question": "Which of the following is not true for myelinated nerve fibers:",
    "opa": "Impulse through myelinated fibers is slower than non-myelinated fibers",
    "opb": "Membrane currents are generated at nodes of Ranvier", 
    "opc": "Saltatory conduction of impulses is seen",
    "opd": "Local anesthesia is effective only when the nerve is not covered by myelin sheath",
    "cop": 1,
    "exp": "Myelinated nerve fibers conduct impulses faster than non-myelinated fibers due to saltatory conduction."
}

In [ ]:
def format_prompt(sample):
    """Format medical question for DeepSeek R1"""
    question = sample.get('question', '')
    opa = sample.get('opa', '')
    opb = sample.get('opb', '')
    opc = sample.get('opc', '')
    opd = sample.get('opd', '')
    explanation = sample.get('exp', '') or ''
    
    # Enhanced system prompt with explanation integration
    system_prompt = """You are an expert medical AI assistant that provides detailed reasoning for medical questions. 
You must think step-by-step through each medical question, considering relevant medical knowledge, differential diagnoses, 
pathophysiology, clinical presentation, and diagnostic criteria before arriving at your final answer.

Please provide your reasoning enclosed in <think></think> tags, followed by your final answer as a single letter (A, B, C, or D).

In your reasoning, consider:
1. Key medical concepts and terminology
2. Pathophysiology and disease mechanisms
3. Clinical presentation and symptoms
4. Differential diagnoses
5. Diagnostic criteria and methods
6. Treatment implications if relevant"""
    
    # Format the question with options
    formatted_prompt = f"""{system_prompt}

Medical Question: {question}

Options:
A) {opa}
B) {opb}
C) {opc}
D) {opd}"""
    
    # Always include explanation to improve reasoning quality
    if explanation.strip():
        formatted_prompt += f"""

Reference Explanation: {explanation}

Use this explanation to guide your reasoning process, but provide your own step-by-step analysis."""
    
    formatted_prompt += "\n\nPlease provide your detailed medical reasoning in <think></think> tags and then state your final answer."
    
    return formatted_prompt

In [6]:
async def test_deepseek_api(prompt):
    """Test API call"""
    api_key = os.getenv('OPENROUTER_API_KEY') or os.getenv('openrouter_api_key')
    
    if not api_key:
        return "❌ API key not found"
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/test",
        "X-Title": "DeepSeek Test"
    }
    
    payload = {
        "model": "deepseek/deepseek-r1",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1000,
        "temperature": 0.1
    }
    
    try:
        async with aiohttp.ClientSession() as session:
            async with session.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=aiohttp.ClientTimeout(total=30)
            ) as response:
                
                if response.status == 200:
                    data = await response.json()
                    return data['choices'][0]['message']['content']
                else:
                    error_text = await response.text()
                    return f"❌ API Error {response.status}: {error_text}"
                    
    except Exception as e:
        return f"❌ Request failed: {e}"

In [7]:
def extract_thinking_and_answer(response):
    """Extract thinking and final answer"""
    # Extract <think></think> content
    think_pattern = r'<think>(.*?)</think>'
    think_match = re.search(think_pattern, response, re.DOTALL)
    thinking = think_match.group(1).strip() if think_match else None
    
    # Extract answer after </think>
    if think_match:
        after_think = response[think_match.end():].strip()
    else:
        after_think = response.strip()
    
    # Look for final answer (A, B, C, D)
    answer_patterns = [
        r'\b([ABCD])\b',
        r'answer\s*:?\s*([ABCD])',
        r'final\s*answer\s*:?\s*([ABCD])',
    ]
    
    final_answer = None
    for pattern in answer_patterns:
        match = re.search(pattern, after_think, re.IGNORECASE)
        if match:
            final_answer = match.group(1).upper()
            break
    
    return thinking, final_answer

def check_correctness(predicted_answer, correct_option):
    """Check if answer is correct"""
    if not predicted_answer:
        return False
    
    option_map = {1: 'A', 2: 'B', 3: 'C', 4: 'D'}
    correct_letter = option_map.get(correct_option)
    
    return predicted_answer.upper() == correct_letter


In [8]:
# Test the complete pipeline
async def run_test():
    print("🧪 Testing DeepSeek R1 API and extraction logic...")
    print("=" * 50)
    
    # 1. Format prompt
    prompt = format_prompt(test_sample)
    print("📝 Formatted prompt:")
    print(prompt[:300] + "..." if len(prompt) > 300 else prompt)
    print("\n" + "=" * 50)
    
    # 2. Call API
    print("🔄 Calling DeepSeek R1 API...")
    response = await test_deepseek_api(prompt)
    
    if response.startswith("❌"):
        print(response)
        return
    
    print("✅ API Response:")
    print(response)
    print("\n" + "=" * 50)
    
    # 3. Extract thinking and answer
    thinking, predicted_answer = extract_thinking_and_answer(response)
    
    print("🧠 Extracted thinking:")
    print(thinking if thinking else "❌ No thinking found")
    print(f"\n🎯 Predicted answer: {predicted_answer}")
    print(f"✅ Correct answer: {test_sample['cop']} (Option {['A', 'B', 'C', 'D'][test_sample['cop']-1]})")
    
    # 4. Check correctness
    is_correct = check_correctness(predicted_answer, test_sample['cop'])
    print(f"🏆 Is correct: {is_correct}")
    
    print("\n" + "=" * 50)
    print("🎉 Test complete!")

# Run the test
await run_test()

🧪 Testing DeepSeek R1 API and extraction logic...
📝 Formatted prompt:
You are an expert medical AI assistant. Think step-by-step through medical questions.

Please provide your reasoning in <think></think> tags, then give your final answer as a single letter (A, B, C, or D).

Medical Question: Which of the following is not true for myelinated nerve fibers:

Options:
A...

🔄 Calling DeepSeek R1 API...
✅ API Response:


🧠 Extracted thinking:
❌ No thinking found

🎯 Predicted answer: None
✅ Correct answer: 1 (Option A)
🏆 Is correct: False

🎉 Test complete!
